In [1]:
#GPU CHECK
!nvidia-smi


Fri May 29 11:10:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.95                 Driver Version: 581.95         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   56C    P3             16W /   41W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# PYTORCH & CUDA CHECK
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch Version: 2.5.1+cu121
CUDA Available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [6]:
# INSTALL DEPENDENCIES
!pip install torch torchvision torchaudio
!pip install -r requirement.txt

# YOLOX
%cd experiments/external/YOLOX
!pip install -r requirements.txt
!pip install -e .

# deep-person-reid
%cd ../deep-person-reid
!pip install -r requirements.txt
!pip install -e .

# fast_reid
%cd ../fast_reid
!pip install -r docs/requirements.txt

# back to DiffMOT-main
%cd ../../..

# YOLOv8
!pip install ultralytics

print("ALL INSTALLED")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[WinError 3] The system cannot find the path specified: 'experiments/external/YOLOX'
C:\Users\vansh\VANSHIKASOHAL\DiffMOT-main



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirement.txt'
C:\Users\vansh\gpu_env\lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Obtaining file:///C:/Users/vansh/VANSHIKASOHAL/DiffMOT-main/datasets
[WinError 2] The system cannot find the file specified: '../deep-person-reid'
C:\Users\vansh\VANSHIKASOHAL\DiffMOT-main



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: file:///C:/Users/vansh/VANSHIKASOHAL/DiffMOT-main/datasets does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


...


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: file:///C:/Users/vansh/VANSHIKASOHAL/DiffMOT-main/datasets does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


C:\Users\vansh



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'docs/requirements.txt'
C:\Users\vansh\gpu_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


ALL INSTALLEDRequirement already satisfied: ultralytics in .\AppData\Local\Programs\Python\Python313\Lib\site-packages (8.4.11)




[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# DATA PREPROCESSING
# Step 1: convert_mot17_to_framewise.py - make det_framewise for all sequences
#Due to some typo issues in MOT17-04-FRCNN det_framewise was not created for this sequence so
# Step 2: createdetfor4.py  was created it creates det_framewise for MOT17-04-FRCNN only

import os
print("Checking det_framewise folders...")
train_path = "datasets/MOT17/train"
for seq in os.listdir(train_path):
    det_fw = os.path.join(train_path, seq, "det_framewise")
    if os.path.exists(det_fw):
        print(f"{seq} - det_framewise exists")
    else:
        print(f" {seq} - det_framewise MISSING")

Checking det_framewise folders...
 desktop.ini - det_framewise MISSING
MOT17-02-FRCNN - det_framewise exists
MOT17-04-FRCNN - det_framewise exists
MOT17-05-FRCNN - det_framewise exists
MOT17-09-FRCNN - det_framewise exists
MOT17-10-FRCNN - det_framewise exists
MOT17-11-FRCNN - det_framewise exists
MOT17-13-FRCNN - det_framewise exists


In [12]:
# Pretrained Models already present locally:
# pretrained/motion/MOT_epoch800.pt  - Motion model
# pretrained/reid/mot17_sbs_S50.pth  - ReID model

In [43]:
#Doing detections using YOLO
import os
import numpy as np
import torch
from ultralytics import YOLO

# go to project folder
#os.chdir("DiffMOT-main")

# CONFIG
MODEL_WEIGHTS = "yolov8n.pt"#check
CONF_THRESH   = 0.3
IOU_THRESH    = 0.45
IMG_SIZE      = 1280
PERSON_CLASS  = 0

# device selection
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = YOLO(MODEL_WEIGHTS)
model.to(device)

# MOT17 sequences
seqs = ['MOT17-02-FRCNN','MOT17-04-FRCNN','MOT17-05-FRCNN','MOT17-09-FRCNN','MOT17-10-FRCNN','MOT17-11-FRCNN','MOT17-13-FRCNN']

for seq in seqs:
    img_dir  = f"datasets/mot/train/{seq}/img1"
    det_dir  = f"datasets/mot/train/{seq}/det"
    det_path = f"{det_dir}/det.txt"

    if not os.path.exists(img_dir):
        print(f"Skipping {seq}, folder not found")
        continue

    os.makedirs(det_dir, exist_ok=True)

    frames = sorted(os.listdir(img_dir))
    print(f"\n[{seq}] Running YOLOv8 on {len(frames)} frames...")

    lines = []

    for frame_file in frames:
        frame_id = int(os.path.splitext(frame_file)[0])
        img_path = os.path.join(img_dir, frame_file)

        results = model.predict(
            source  = img_path,
            conf    = CONF_THRESH,
            iou     = IOU_THRESH,
            imgsz   = IMG_SIZE,
            classes = [PERSON_CLASS],
            verbose = False,
            device  = device
        )

        boxes = results[0].boxes

        if boxes is not None and len(boxes) > 0:
            xyxy  = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()

            for (x1, y1, x2, y2), conf in zip(xyxy, confs):
                w = x2 - x1
                h = y2 - y1
                lines.append(f"{frame_id},-1,{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1")

    with open(det_path, "w") as f:
        f.write("\n".join(lines))

    print(f"Saved {len(lines)} detections to {det_path}")

print("YOLOv8 detections complete")


Using device: cuda

[MOT17-02-FRCNN] Running YOLOv8 on 600 frames...
Saved 7643 detections to datasets/mot/train/MOT17-02-FRCNN/det/det.txt

[MOT17-04-FRCNN] Running YOLOv8 on 1050 frames...
Saved 29088 detections to datasets/mot/train/MOT17-04-FRCNN/det/det.txt

[MOT17-05-FRCNN] Running YOLOv8 on 837 frames...
Saved 5316 detections to datasets/mot/train/MOT17-05-FRCNN/det/det.txt

[MOT17-09-FRCNN] Running YOLOv8 on 525 frames...
Saved 5153 detections to datasets/mot/train/MOT17-09-FRCNN/det/det.txt

[MOT17-10-FRCNN] Running YOLOv8 on 654 frames...
Saved 9006 detections to datasets/mot/train/MOT17-10-FRCNN/det/det.txt

[MOT17-11-FRCNN] Running YOLOv8 on 900 frames...
Saved 7977 detections to datasets/mot/train/MOT17-11-FRCNN/det/det.txt

[MOT17-13-FRCNN] Running YOLOv8 on 750 frames...
Saved 5887 detections to datasets/mot/train/MOT17-13-FRCNN/det/det.txt
YOLOv8 detections complete


In [44]:
#VERIFICATIONS OF DETECTIONS OUTPUTS
import os
for seq in seqs:
    path=f"datasets/mot/train/{seq}/det/det.txt"
    print(seq,"->",os.path.exists(path))

MOT17-02-FRCNN -> True
MOT17-04-FRCNN -> True
MOT17-05-FRCNN -> True
MOT17-09-FRCNN -> True
MOT17-10-FRCNN -> True
MOT17-11-FRCNN -> True
MOT17-13-FRCNN -> True


In [13]:
#VERIFICATION
import os

#os.chdir("DiffMOT-main")

seqs = ['MOT17-02-FRCNN','MOT17-04-FRCNN','MOT17-05-FRCNN','MOT17-09-FRCNN','MOT17-10-FRCNN','MOT17-11-FRCNN','MOT17-13-FRCNN']

print("Detection file check:")

for seq in seqs:
    det_path = f"datasets/MOT17/train/{seq}/det/det.txt"

    if os.path.exists(det_path):
        lines = open(det_path).readlines()
        print(f"{seq}: {len(lines)} detections")

        if len(lines) > 0:
            print("Sample:", lines[0].strip())
        else:
            print("EMPTY FILE")

    else:
        print(f"{seq}: det.txt NOT FOUND")


Detection file check:
MOT17-02-FRCNN: 7643 detections
Sample: 1,-1,1338.96,416.53,167.61,374.16,0.8653,-1,-1,-1
MOT17-04-FRCNN: 29088 detections
Sample: 1,-1,103.05,548.87,76.52,246.55,0.8761,-1,-1,-1
MOT17-05-FRCNN: 5316 detections
Sample: 1,-1,17.99,148.51,73.29,190.32,0.8976,-1,-1,-1
MOT17-09-FRCNN: 5153 detections
Sample: 1,-1,1288.37,460.81,71.61,198.44,0.8149,-1,-1,-1
MOT17-10-FRCNN: 9006 detections
Sample: 1,-1,1475.04,419.19,75.20,169.65,0.8781,-1,-1,-1
MOT17-11-FRCNN: 7977 detections
Sample: 1,-1,0.00,26.08,347.06,989.28,0.8907,-1,-1,-1
MOT17-13-FRCNN: 5887 detections
Sample: 1,-1,0.08,661.50,49.74,222.84,0.8220,-1,-1,-1


In [14]:
# CONFIG UPDATE FOR TRACKING
import yaml
import os

config_path = "configs/mot17_test.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Original config:")
print(yaml.dump(config))

config["det_dir"]    = "datasets/MOT17/train"
config["reid_dir"]   = "pretrained/reid"
config["model_path"] = "pretrained/motion/MOT_epoch800.pt"
config["info_dir"]   = "datasets/MOT17/train"
config["save_dir"]   = "outputs/mot17"
config["device"]     = "cuda"
config["gpus"]       = [0]

# Thresholds
config["high_thres"]  = 0.6
config["low_thres"]   = 0.1
config["w_assoc_emb"] = 2.2
config["aw_param"]    = 1.7
config["batch_size"]  = 512

# Save config
with open(config_path, "w") as f:
    yaml.dump(config, f)

print("Updated config saved!\n")
with open(config_path) as f:
    print(f.read())

Original config:
augment: true
aw_param: 1.7
batch_size: 2048
data_dir: /mnt/8T/home/estar/data/DanceTrack/trackers_gt_GSI
det_dir: /mnt/8T/home/estar/data/DanceTrack/detections/val
device: cuda
diffnet: HMINet
encoder_dim: 256
epochs: 800
eps: 0.001
eval_at: 800
eval_device: None
eval_every: 20
eval_expname: mot_ddm_1000_deeper
eval_mode: true
gpus:
- 0
- 1
- 2
- 3
high_thres: 0.6
info_dir: /mnt/8T/home/estar/data/DanceTrack/val
interval: 5
low_thres: 0.1
lr: 0.0001
preprocess_workers: 16
reid_dir: /home/estar/lwy/DiffMOT/cache/embeddings/
save_dir: /mnt/8T/home/estar/data/DanceTrack/results/val/yolox_m_lt_ddm_1000eps_deeper_800_1rev
seed: 123
tf_layer: 3
w_assoc_emb: 2.2

Updated config saved!

augment: true
aw_param: 1.7
batch_size: 512
data_dir: /mnt/8T/home/estar/data/DanceTrack/trackers_gt_GSI
det_dir: datasets/MOT17/train
device: cuda
diffnet: HMINet
encoder_dim: 256
epochs: 800
eps: 0.001
eval_at: 800
eval_device: None
eval_every: 20
eval_expname: mot_ddm_1000_deeper
eval_mode:

In [59]:
print(open("configs/mot17_test.yaml").read())

augment: true
aw_param: 1.7
batch_size: 512
data_dir: /mnt/8T/home/estar/data/DanceTrack/trackers_gt_GSI
det_dir: datasets/mot/train
device: cuda
diffnet: HMINet
encoder_dim: 256
epochs: 800
eps: 0.001
eval_at: 800
eval_device: None
eval_every: 20
eval_expname: mot_ddm_1000_deeper
eval_mode: true
gpus:
- 0
high_thres: 0.6
info_dir: datasets/mot/train
interval: 5
low_thres: 0.1
lr: 0.0001
model_path: pretrained/motion/MOT_epoch800.pt
preprocess_workers: 16
reid_dir: pretrained/reid
save_dir: outputs/mot17
seed: 123
tf_layer: 3
w_assoc_emb: 2.2



In [10]:
#INSTALLATION
pip install -r requirement.txt

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached cython_bbox-0.1.5.tar.gz (4.4 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requ

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [7 lines of output]
  running bdist_wheel
  running build
  running build_ext
  Compiling src/cython_bbox.pyx because it changed.
  [1/1] Cythonizing src/cython_bbox.pyx
  building 'cython_bbox' extension
  error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
  [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for cython-bbox
error: failed-wheel-build-for-install

Failed to build installable wheels for some pyproject.toml based projects

cython-bbox


In [4]:
#VERIFICATION FOR PRETRAINED MODEL FILE
import os

model_path = "pretrained/motion/MOT_epoch800.pt"

print("Model exists:", os.path.exists(model_path))

# list pretrained folder
print("\nFiles in pretrained/motion:")
print(os.listdir("pretrained/motion"))

Model exists: True

Files in pretrained/motion:
['MOT_epoch800.pt']


In [105]:
import torch

path = "pretrained/motion/MOT_epoch800.pt"

model = torch.load(path, map_location="cpu")
print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'dict'>


In [23]:
!python main.py \
--dataset mot \
--data_dir datasets \
--dataset_name MOT17 \
--mode test \
--device cuda \
--batch_size 1 \
--exp_name mot_ddm_1000_deeper \
--detector FRCNN

> Train Dataset built!


2026-04-17 22:54:44 [DEBUG]: matplotlib data path: C:\Users\vansh\diffmot_env\Lib\site-packages\matplotlib\mpl-data
2026-04-17 22:54:44 [DEBUG]: CONFIGDIR=C:\Users\vansh\.matplotlib
2026-04-17 22:54:44 [DEBUG]: interactive is False
2026-04-17 22:54:44 [DEBUG]: platform is win32
2026-04-17 22:54:44 [DEBUG]: CACHEDIR=C:\Users\vansh\.matplotlib
2026-04-17 22:54:44 [DEBUG]: Using fontManager instance from C:\Users\vansh\.matplotlib\fontlist-v390.json
C:\Users\vansh\diffmot_env\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
usage: main.py [-h] [--config CONFIG] [--dataset DATASET]
main.py: error: unrecognized arguments: --data_dir datasets --dataset_name MOT17 --mode test --device cuda --batch_size 1 --exp_name mot_ddm_1000_deeper --detector FRCNN


In [20]:
!pip install fastreid


  Using cached fastreid-1.4.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached faiss_cpu-1.14.2-cp313-cp313-win_amd64.whl.metadata (7.8 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
INFO: pip is looking at multiple versions of fastreid to determine which version is compatible with other requirements. This could take a while.


ERROR: Could not find a version that satisfies the requirement torch==1.13.1 (from fastreid) (from versions: 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch==1.13.1


In [7]:

import sys
import os

# project root
project_path = r"C:\Users\vansh\VANSHIKASOHAL\DiffMOT-main"

# add all required paths
sys.path.insert(0, project_path)
sys.path.insert(0, os.path.join(project_path, "external"))
sys.path.insert(0, os.path.join(project_path, "external", "fast_reid"))
sys.path.insert(0, os.path.join(project_path, "external", "fast_reid", "fastreid"))

print("PATH SET ")

PATH SET 


In [8]:
#VERIFICATION
import fastreid
print("FASTREID INSTALLED SUCCESSFULLY")

FASTREID INSTALLED SUCCESSFULLY


In [12]:
#VERIFICATION
from fastreid.config import get_cfg
from external.adaptors.fastreid_adaptor import FastReID

print("ALL IMPORTS WORKING ")

ALL IMPORTS WORKING 


In [13]:
#VERIFICATION
from external.adaptors.fastreid_adaptor import FastReID

model = FastReID()
print("MODEL LOADED")

MODEL LOADED


In [15]:
#VERIFICATION
import torch
print("TORCH INSTALLED SUCCESFFULLY !!")

TORCH INSTALLED SUCCESFFULLY !!


In [2]:
#VERIFICATION
import os

paths = {
    "Motion Model": "pretrained/motion/MOT_epoch800.pt",
    "ReID Model": "pretrained/reid/mot17_sbs_S50.pth",
    "Detections": "datasets/MOT17/train/MOT17-09-FRCNN/det/det.txt"
}

for name, path in paths.items():
    print(f"\n{name}:")
    print("Path:", path)
    print("Exists:", os.path.exists(path))
    
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"Size: {size:.2f} MB")


Motion Model:
Path: pretrained/motion/MOT_epoch800.pt
Exists: True
Size: 168.32 MB

ReID Model:
Path: pretrained/reid/mot17_sbs_S50.pth
Exists: True
Size: 302.89 MB

Detections:
Path: datasets/MOT17/train/MOT17-09-FRCNN/det/det.txt
Exists: True
Size: 0.25 MB


In [ ]:
#TRACKING(RAN IT IN CMD MODE) ALL INSTALLATIONS RELATED TO IT WERE DONE IN CMD MODE
python main.py --config configs/mot.yaml --dataset MOT17


In [ ]:
#EVALUATION ON TRACKED OUTPUTS ALL INSTALLATIONS RELATED TO IT WERE DONE IN CMD MODE
#python -m trackeval.scripts.run_mot_challenge ^
#--SPLIT_TO_EVAL train ^
#--METRICS HOTA Identity ^
#--GT_FOLDER datasets/mot/train ^
#--TRACKERS_FOLDER outputs/mot17 ^
#--TRACKER_SUB_FOLDER "" ^
#--USE_PARALLEL False ^
#--NUM_PARALLEL_CORES 1 ^
#--PRINT_RESULTS True ^
#--OUTPUT_SUMMARY True
python yoloeval.py